# Detection Pipeline (PKU-MMD / TSU) — Kaggle (Input từ Dataset)

**Chiến lược I/O tối ưu trên Kaggle:**
- **Đầu vào (Video & Annotation):** Đã có sẵn hoàn toàn trong `/kaggle/input/` (Kaggle Datasets).
- **Ghi output:** Ra `/kaggle/working/outputs_detection` (SSD nhanh)
- **Sync sau mỗi lô:** `rclone copy` tự động upload output lên Drive sau mỗi lô (để lưu trữ an toàn)

> ⚠️ **Lưu ý Kaggle:**
> - Mỗi session giới hạn **12 tiếng**.
> - Phải **BẬT** Internet trong Kaggle Settings (để rclone upload được).
> - Hãy đảm bảo đã **Add Data** cả Video Dataset và Annotation Dataset vào notebook này.

**Thứ tự chạy:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → **8**


In [ ]:
# Cell 1: Lay code moi nhat tu GitHub
import os
REPO   = "https://github.com/tuan8p/Skeleton-EAA-Pose.git"
BRANCH = "dai"
WORKDIR = "/kaggle/working/Skeleton-EAA-Pose"
if not os.path.isdir(WORKDIR):
    !git clone -b {BRANCH} {REPO} {WORKDIR}
else:
    !git -C {WORKDIR} pull origin {BRANCH}
%cd {WORKDIR}

In [ ]:
# Cell 2: Cai dat thu vien + rclone
# Cai rclone (Kaggle chua co san, phai cai thu cong)
!curl https://rclone.org/install.sh | bash 2>/dev/null || true
!which rclone && rclone version

# Cai thu vien Python
!pip install -q -r requirements.txt

In [ ]:
# Cell 3: Cấu hình thư mục Input từ Kaggle & rclone (để upload output)
import os, subprocess, json
from kaggle_secrets import UserSecretsClient

# ==========================================================================
# 1. CHỌN DATASET VÀ ĐIỀN 4 ĐƯỜNG DẪN KAGGLE VÀO ĐÂY:
# ==========================================================================
DATASET = "TSU"   # Chọn "PKU" hoặc "TSU"

PKU_PATHS = {
    "video_dir":      "/kaggle/input/datasets/tuan8p/pku-rgb",
    "annotation_dir": "/kaggle/input/datasets/tuan8p/pku-annotation",
}

TSU_PATHS = {
    "video_dir":      "/kaggle/input/datasets/tuan8p/tsu-rgb/Videos_mp4",
    "annotation_dir": "/kaggle/input/datasets/tuan8p/tsu-annotation/Annotation_v1.0",
}

# --- Tự động chọn thư mục tương ứng --- 
if DATASET == "PKU":
    VIDEO_INPUT_DIR = PKU_PATHS["video_dir"]
    ANN_INPUT_DIR   = PKU_PATHS["annotation_dir"]
else:
    VIDEO_INPUT_DIR = TSU_PATHS["video_dir"]
    ANN_INPUT_DIR   = TSU_PATHS["annotation_dir"]

# --- Đọc RCLONE_TOKEN từ Kaggle Secrets (Dành cho việc upload Output) ---
try:
    secrets = UserSecretsClient()
    RCLONE_TOKEN = secrets.get_secret("RCLONE_TOKEN")
    GDRIVE_REMOTE = "gdrive"
    rclone_conf_dir = os.path.expanduser("~/.config/rclone")
    os.makedirs(rclone_conf_dir, exist_ok=True)
    rclone_conf = f"""[{GDRIVE_REMOTE}]
type = drive
token = {RCLONE_TOKEN}
scope = drive
"""
    conf_path = os.path.join(rclone_conf_dir, "rclone.conf")
    with open(conf_path, "w") as f:
        f.write(rclone_conf)
    print(f"[OK] Đã tạo rclone config tại: {conf_path}")
except Exception as e:
    print(f"[WARN] Không thể cấu hình rclone tự động: {e}")

# --- Cài đặt thư mục làm việc cục bộ (để chứa kết quả đầu ra) ---
LOCAL_OUTPUT   = "/kaggle/working/outputs_detection"
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

# --- Đọc danh sách file từ thư mục Input ---
print(f"\nĐang đọc danh sách video từ: {VIDEO_INPUT_DIR}")
if not os.path.isdir(VIDEO_INPUT_DIR):
    print(f"[LỖI] Thư mục không tồn tại: {VIDEO_INPUT_DIR}. Vui lòng kiểm tra lại Data bên cột phải.")
    _all_files = []
else:
    _all_files = []
    for fname in os.listdir(VIDEO_INPUT_DIR):
        if fname.endswith(".mp4") or fname.endswith(".avi"):
            fpath = os.path.join(VIDEO_INPUT_DIR, fname)
            _all_files.append((fname, os.path.getsize(fpath)))

_all_files.sort(key=lambda x: x[0])  # sort theo tên

START, END = 350, 400
_slice_end = END if (END is not None and END > 0) else len(_all_files)
VIDEO_FILE_LIST = _all_files[START:_slice_end]  # list[(name, size_bytes)]

print(f"[OK] Tổng số video tìm thấy: {len(_all_files)}")
print(f"[OK] Sẽ xử lý {len(VIDEO_FILE_LIST)} video (index {START} đến {_slice_end})")
if VIDEO_FILE_LIST:
    print(f"[OK] 3 video đầu tiên: {[v[0] for v in VIDEO_FILE_LIST[:3]]}")

print(f"\nĐang kiểm tra thư mục Annotation: {ANN_INPUT_DIR}")
if not os.path.isdir(ANN_INPUT_DIR):
    print(f"[LỖI] Thư mục Annotation không tồn tại: {ANN_INPUT_DIR}")
else:
    ann_files = os.listdir(ANN_INPUT_DIR)
    print(f"[OK] Tìm thấy {len(ann_files)} file annotation.")

def get_disk_free_gb(path="/kaggle/working"):
    import shutil
    _, _, free = shutil.disk_usage(path)
    return free / 1e9

print(f"\n[Config] Disk hiện tại còn trống: {get_disk_free_gb():.1f} GB")


In [ ]:
# Cell 4: Cấu hình cài đặt Pipeline

# -----------------------------------------------------
# LƯU Ý: PKU_PATHS, TSU_PATHS, DATASET đã được khai báo ở Cell 3.
# -----------------------------------------------------

DEPTH_PATHS = {
    "PKU": "/kaggle/working/depth_batch",
    "TSU": "/kaggle/working/depth_batch",
}

# Output ghi ra LOCAL SSD của Kaggle (I/O nhanh nhất)
LOCAL_OUTPUT = "/kaggle/working/outputs_detection"

# Drive path (dùng để rclone copy output lên Drive lưu trữ)
DRIVE_OUTPUT_REMOTE = "ĐACN-TN_datasets/ĐATN/rawdatasets/outputs_detection"  # path trên Drive (không có gdrive:)

# ==========================================
# CẤU HÌNH PIPELINE
# ==========================================
SETTINGS = {
    "yolo.model_name": "yolo26x.pt",
    "yolo.tracker": "iou",
    "runtime.num_workers": 8,            # 2xT4: 8 luồng song song
    "yolo.batch_size": 64,               # 2xT4: batch 64
    "yolo.conf_threshold": 0.2,
    "yolo.iou_threshold": 0.7,
    "detection.retry.max_retries": 2,
    "detection.retry.imgsz_retry": 1280,
    "detection.retry.tta": False,
    "detection.retry.clahe_on_retry": True,
    "chunk.max_actions_per_chunk": 5,
    "tsu.video_ext": ".mp4",
    "tsu.has_header": True,
    "tsu.event_column": 0,
    "tsu.start_column": 1,
    "tsu.end_column": 2,
    "tsu.event_map": {},
    "tsu.event_mapping_path": "/kaggle/input/datasets/tuan8p/tsu-annotation/event_mapping.csv",
}

USE_DEPTH = False


In [ ]:
# Cell 5: Kiem tra disk space Kaggle truoc khi chay
import shutil, os

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Disk /kaggle/working: Total={total//1e9:.1f}GB | Used={used//1e9:.1f}GB | Free={free//1e9:.1f}GB")
if free < 3 * 1e9:
    print("[WARN] Sap het disk! Xoa bo cache cu neu co.")
    !rm -rf /kaggle/working/outputs_detection/__pycache__

os.makedirs(LOCAL_OUTPUT, exist_ok=True)
print(f"Local output dir: {LOCAL_OUTPUT}")

# Kiem tra doc duoc data tu Drive chua
paths = PKU_PATHS if DATASET == "PKU" else TSU_PATHS
video_dir = paths["video_dir"]
ann_dir   = paths["annotation_dir"]

assert os.path.isdir(video_dir), f"Khong tim thay video_dir: {video_dir}"
assert os.path.isdir(ann_dir),   f"Khong tim thay annotation_dir: {ann_dir}"

videos = sorted(os.listdir(video_dir))
TOTAL_VIDEOS = len(videos)
if END is None or END > TOTAL_VIDEOS:
    END = TOTAL_VIDEOS
print(f"[{DATASET}] Tim thay {TOTAL_VIDEOS} files. Chay tu {START} den {END}")
print(f"3 file dau: {videos[:3]}")
print(f"[OK] Tat ca path hop le, san sang chay pipeline!")

In [ ]:
# Cell 6: (Tuy chon) Setup Kaggle session keepalive
# Kaggle tu dong ngat ket noi sau ~1h khong co tuong tac.
# Script nay in tien trinh dinh ky de Kaggle biet session van hoat dong.
import threading, time, datetime

_keepalive_stop = threading.Event()

def _keepalive_worker(interval_min=10):
    while not _keepalive_stop.wait(interval_min * 60):
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        # In ra de Kaggle biet session con hoat dong
        output_files = []
        import os
        for root, _, files in os.walk(LOCAL_OUTPUT):
            output_files.extend(files)
        print(f"[{ts}] Keepalive ping | Output files: {len(output_files)}", flush=True)

_keepalive_thread = threading.Thread(target=_keepalive_worker, daemon=True)
_keepalive_thread.start()
print("[OK] Keepalive thread da khoi dong (ping moi 10 phut)")

In [ ]:
# Cell 7: Chạy pipeline theo chunk và Sync Output lên Drive
# ==========================================================================
import torch, os, subprocess, datetime, multiprocessing as mp
from src.config_manager import ConfigManager
from src.detection_pipeline import DetectionPipeline

# Để tránh lỗi multiprocessing (spawn) trên Kaggle, ta ghi hàm worker ra một file python riêng
worker_code = """import os
from src.config_manager import ConfigManager
from src.detection_pipeline import DetectionPipeline

def worker_process(gpu_id, names_chunk, dataset_name, total_n):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    cfg_w = ConfigManager("config.runtime.yaml")
    orch  = DetectionPipeline(cfg=cfg_w)
    for vname in names_chunk:
        try:
            from src.detection_logger import DetectionLogger
            logger_mock = DetectionLogger(cfg_w, dataset_name, 0, total_n)
            stats = orch.process_video(vname, logger_mock, disable_pbar=True)
            logger_mock.write_video_log(stats, cfg_w.as_dict())
            logger_mock.append_batch_summary(stats)
            print(f"    [GPU {gpu_id}] Xong: {vname} ok={stats.ok_frames}/{stats.total_frames}")
        except Exception as exc:
            import traceback
            print(f"    [GPU {gpu_id}] LỖI {vname}: {exc}")
            print(traceback.format_exc())
"""
with open("kaggle_worker.py", "w", encoding="utf-8") as f:
    f.write(worker_code)

import kaggle_worker
import importlib
importlib.reload(kaggle_worker) # Đảm bảo nạp lại nếu file bị sửa

def _sync_output_to_drive():
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    print(f"[{ts}] Đang sync output lên Drive...")
    r = subprocess.run(
        ["rclone", "copy", LOCAL_OUTPUT,
         f"gdrive:{DRIVE_OUTPUT_REMOTE}",
         "--transfers", "8",
         "--drive-chunk-size", "128M"],
        capture_output=False, text=True
    )
    if r.returncode == 0:
        print(f"[{ts}] Sync thành công!")
    else:
        print(f"[WARN] rclone exit code: {r.returncode} — Tiếp tục...")

def _run_pipeline_for_names(video_names_in_batch):
    num_gpus = torch.cuda.device_count()
    n = len(video_names_in_batch)
    print(f"  -> Phát hiện {num_gpus} GPU, xử lý {n} video trong lô...")

    if num_gpus > 1:
        mid = (n + 1) // 2
        chunk0 = video_names_in_batch[:mid]
        chunk1 = video_names_in_batch[mid:]
        mp.set_start_method('spawn', force=True)
        p0 = mp.Process(target=kaggle_worker.worker_process, args=(0, chunk0, DATASET, n))
        p1 = mp.Process(target=kaggle_worker.worker_process, args=(1, chunk1, DATASET, n))
        p0.start(); p1.start()
        p0.join();  p1.join()
    else:
        kaggle_worker.worker_process(0, video_names_in_batch, DATASET, n)

# ── Nạp config ──
cfg = ConfigManager("config.yaml")
cfg.set("dataset", DATASET)
paths = PKU_PATHS if DATASET == "PKU" else TSU_PATHS
for k, v in paths.items():
    cfg.set(f"paths.{k}", v)
cfg.set("detection.output_dir", LOCAL_OUTPUT)
cfg.set("paths.depth_dir", DEPTH_PATHS[DATASET])
cfg.set("depth.enabled", USE_DEPTH)
for k, v in SETTINGS.items():
    cfg.set(k, v)
cfg.save("config.runtime.yaml")

# ── Chia nhỏ list để sync định kỳ (vd: cứ 50 video sync 1 lần) ──
CHUNK_SIZE = 50
video_chunks = [VIDEO_FILE_LIST[i:i + CHUNK_SIZE] for i in range(0, len(VIDEO_FILE_LIST), CHUNK_SIZE)]

print(f"[Hệ thống] Tổng {len(VIDEO_FILE_LIST)} video chia thành {len(video_chunks)} lô để sync định kỳ.")

for batch_no, batch in enumerate(video_chunks, start=1):
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    print(f"\n{'='*60}")
    print(f"[{ts}] === Lô {batch_no}/{len(video_chunks)} | {len(batch)} video ===")
    print(f"{'='*60}")

    video_names = [os.path.splitext(f[0])[0] for f in batch]
    _run_pipeline_for_names(video_names)
    _sync_output_to_drive()

print(f"\n{'='*60}")
print(f"[HOÀN TẤT] Đã xử lý tất cả video!")
print(f"Output tại: {LOCAL_OUTPUT}")
print(f"Đã sync lên Drive: gdrive:{DRIVE_OUTPUT_REMOTE}")


In [ ]:
# Cell 8: (TUY CHON) Sync thu cong bat cu luc nao
# Sau khi pipeline hoan tat, vong lap Cell 8 da tu dong sync sau moi lo.
# Chay cell nay neu ban muon sync lai thu cong (vd: sau khi resume).
import subprocess, datetime
ts = datetime.datetime.now().strftime('%H:%M:%S')
print(f"[{ts}] Bat dau sync thu cong...")
subprocess.run([
    "rclone", "copy", LOCAL_OUTPUT,
    f"gdrive:{DRIVE_OUTPUT_REMOTE}",
    "--transfers", "8",
    "--drive-chunk-size", "128M",
    "--progress"
], check=False)
print(f"[{ts}] Sync thu cong hoan tat!")


In [ ]:
# Cell 9: (TÙY CHỌN) Ép chạy lại (Re-run) các video cụ thể
# Hướng dẫn: Nhập tên các video muốn chạy lại vào mảng bên dưới (có đuôi hay không đuôi .mp4 đều được)
VIDEOS_TO_RERUN = ['P03T15C03', 'P12T14C05', 'P13T03C04', 'P13T04C04', 'P13T05C04', 'P13T06C04', 'P13T07C04', 'P13T08C02', 'P13T09C01', 'P13T10C01', 'P13T10C02', 'P13T11C01', 'P13T11C02', 'P13T12C01', 'P13T12C02', 'P13T13C01', 'P13T21C06', 'P13T21C07', 'P13T23C03', 'P13T23C06', 'P13T23C07', 'P13T24C04', 'P13T24C05', 'P13T25C04', 'P13T25C05', 'P14T01C04', 'P14T01C05', 'P14T02C04', 'P14T02C05', 'P14T03C04', 'P14T03C05', 'P14T04C04', 'P14T04C05', 'P14T05C04', 'P14T05C05']


import os, json

if VIDEOS_TO_RERUN:
    # Chuẩn hóa tên (bỏ đuôi mở rộng)
    vnames = [os.path.splitext(v)[0] for v in VIDEOS_TO_RERUN]
    print(f"Bắt đầu ép chạy lại {len(vnames)} video: {vnames}")
    
    # 1. Reset trạng thái trong progress.json
    # progress_path = os.path.join(LOCAL_OUTPUT, DATASET, "bboxes", "progress.json")
    # if os.path.exists(progress_path):
    #     with open(progress_path, "r", encoding="utf-8") as f:
    #         prog = json.load(f)
    #     modified = False
    #     for v in vnames:
    #         if v in prog:
    #             del prog[v]
    #             modified = True
    #     if modified:
    #         with open(progress_path, "w", encoding="utf-8") as f:
    #             json.dump(prog, f, indent=1)
    #         print("  -> Đã reset checkpoint trong progress.json")
            
    # 2. Xóa các file kết quả cũ
    for v in vnames:
        old_jsonl = os.path.join(LOCAL_OUTPUT, DATASET, "bboxes", "jsonl", f"{v}.jsonl")
        if os.path.exists(old_jsonl):
            os.remove(old_jsonl)
            print(f"  -> Đã xóa kết quả cũ: {v}.jsonl")
            
    # 3. Chạy lại pipeline cho nhóm video này
    # Lưu ý: Hàm _run_pipeline_for_names đã được định nghĩa ở Cell 7
    _run_pipeline_for_names(vnames)
    
    # 4. Sync kết quả mới lên Google Drive
    # _sync_output_to_drive()
    print("HOÀN TẤT CHẠY LẠI!")
else:
    print("Mảng đang trống, không có video nào cần chạy lại.")


## Ghi chu toi uu I/O cho Kaggle

| Dac diem Kaggle | Cach xu ly |
|---|---|
| **rclone chua co san** | Cell 2 tu dong cai qua `curl` |
| **Khong co browser** | Dung headless token qua Kaggle Secrets |
| **Gioi han 12 tieng/session** | Keepalive thread ping moi 10 phut |
| **Disk /kaggle/working/ = 20GB** | Ghi output local (jsonl nhe ~100MB/session); video KHONG tai ve |
| **Internet On bat buoc** | Phai bat trong Kaggle Settings > Internet |
| **`--vfs-cache-max-size 15G`** | Giu 5GB cho output + overhead |
| **Sync cuoi session** | Tranh nghẽn network khi 4 worker ghi dong thoi |